In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

## CONNEXION POSTGRESQL


In [2]:
USER = "postgres"
PASSWORD = "19981003"
HOST = "localhost"
PORT = "5432"
DATABASE = "darkom_dwh"

engine = create_engine(
    f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
)

print("Connexion réussie")

Connexion réussie


## READ TABLE STAGING

In [3]:
query = "SELECT * FROM staging.annonces_raw"

df = pd.read_sql(query, engine)

print(df.head())
print(df.shape)

  annonce_id date_publication                         titre    ville  \
0  ANO000343       2023-05-05        Terrain moderne Meknès   Meknès   
1  ANO001296       2023-08-20       Terrain moderne Kenitra  Kenitra   
2  ANO001212       2023-06-20     Appartement moderne Oujda    Oujda   
3  ANO001000       2024-01-21  Appartement à vente - Tanger   Tanger   
4  ANO001123       2024-03-19              Beau Villa Rabat    Rabat   

   quartier    type_bien transaction       prix  surface  nb_chambres  \
0    Hamria      Terrain       Vente  811401.04   314.40          0.0   
1    Centre      Terrain    Location    3843.55    36.46          0.0   
2  Hay Qods  Appartement    Location    9190.70    95.54          2.0   
3       NaN  Appartement       Vente   30000.00    93.35          1.0   
4   Souissi        Villa    Location   20844.72   264.22          3.0   

   nb_salles_bain  etage  annee_construction  
0             NaN    0.0              2014.0  
1             0.0    NaN          

## DROP DUPLICATES

In [4]:
df.duplicated().sum()

df.drop_duplicates(inplace=True)

## NULLS 

In [5]:
# 1. SORT & FILL DATES

df = df.sort_values("date_publication").reset_index(drop=True)

df["date_publication"] = (
    df["date_publication"]
    .ffill()
    .bfill()
)

# 2. FILL QUARTIER

df["quartier"] = (
    df.groupby("ville")["quartier"]
    .transform(
        lambda x: x.fillna(
            x.mode().iloc[0] if not x.mode().empty else "Unknown"
        )
    )
)

# 3. EXTRACT TYPE_BIEN

TYPE_MAP = {
    "appartement": "Appartement",
    "villa": "Villa",
    "bureau": "Bureau",
    "terrain": "Terrain",
    "duplex": "Duplex"
}

def extract_type_bien(title):

    title = str(title).lower()

    for key, value in TYPE_MAP.items():

        if key in title:
            return value

    return None


df["type_bien"] = df["type_bien"].fillna(
    df["titre"].apply(extract_type_bien)
)

# 4. FILL TRANSACTION

df["transaction"] = df["transaction"].fillna(
    df["prix"].apply(
        lambda x: "Location" if x <= 30000 else "Vente"
    )
)

# 5. NUMERIC COLUMNS

numeric_cols = [
    "nb_chambres",
    "nb_salles_bain",
    "etage"
]

# Terrain => 0
terrain_mask = df["type_bien"] == "Terrain"

df.loc[terrain_mask, numeric_cols] = (
    df.loc[terrain_mask, numeric_cols]
    .fillna(0)
)

# Fill median by type_bien
for col in numeric_cols:

    df[col] = (
        df.groupby("type_bien")[col]
        .transform(lambda x: x.fillna(x.median()))
    )

# 6. FILL ANNEE_CONSTRUCTION

df["annee_construction"] = df["annee_construction"].fillna(2000)

In [6]:
#Âge estimé du bien immobilier
from datetime import datetime

current_year = datetime.now().year

df["age_bien"] = current_year - df["annee_construction"]

In [7]:
#Catégories de prix
def categorie_prix(x):
    if x < 500000:
        return "Économique"
    elif x < 1500000:
        return "Moyen"
    elif x < 3000000:
        return "Haut standing"
    else:
        return "Luxe"

df["categorie_prix"] = df["prix"].apply(categorie_prix)

In [8]:
#Catégories de surface
def categorie_surface(x):
    if x < 80:
        return "Petit"
    elif x <= 150:
        return "Moyen"
    else:
        return "Grand"

df["categorie_surface"] = df["surface"].apply(categorie_surface)

In [9]:
#Dimensions temporelles
df["date_publication"] = pd.to_datetime(df["date_publication"], errors="coerce")

df["annee_pub"] = df["date_publication"].dt.year
df["mois_pub"] = df["date_publication"].dt.month
df["trimestre_pub"] = df["date_publication"].dt.to_period("Q").astype(str)

In [10]:
df.head()

,annonce_id,date_publication,titre,ville,quartier,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain,etage,annee_construction,age_bien,categorie_prix,categorie_surface,annee_pub,mois_pub,trimestre_pub
0,ANO000708,2023-01-01,Villa moderne Agadir,Agadir,Talborjt,Villa,Vente,2592302.14,248.46,6.0,4.0,0.0,2007.0,19.0,Haut standing,Grand,2023,1,2023Q1
1,ANO001039,2023-01-01,Appartement à vente - Kenitra,Kenitra,Hay Essalam,Appartement,Vente,2155591.49,91.29,2.0,1.0,2.0,2008.0,18.0,Haut standing,Moyen,2023,1,2023Q1
2,ANO000870,2023-01-02,Bureau à vente - Tétouan,Tétouan,M'diq,Bureau,Vente,317773.58,155.88,0.0,0.0,1.0,2020.0,6.0,Économique,Grand,2023,1,2023Q1
3,ANO001275,2023-01-03,Beau Villa Meknès,Meknès,Hamria,Villa,Vente,2177553.48,264.60,4.0,2.0,0.0,2004.0,22.0,Haut standing,Grand,2023,1,2023Q1
4,ANO000883,2023-01-03,Terrain à location - Meknès,Meknès,Ville Nouvelle,Terrain,Location,2755.39,12.00,0.0,0.0,0.0,2000.0,26.0,Économique,Petit,2023,1,2023Q1


In [11]:
df.isnull().sum()

annonce_id            0
date_publication      0
titre                 0
ville                 0
quartier              0
type_bien             0
transaction           0
prix                  0
surface               0
nb_chambres           0
nb_salles_bain        0
etage                 0
annee_construction    0
age_bien              0
categorie_prix        0
categorie_surface     0
annee_pub             0
mois_pub              0
trimestre_pub         0
dtype: int64

In [12]:
import os

os.makedirs("data", exist_ok=True)

In [13]:
df.to_csv(
    r"C:\Users\USB\Desktop\End-to-End-Data-Pipeline-pour-l-analyse-du-march-immobilier-marocain-main\cleaning\data\darkoum_annonces_clean.csv",
    index=False
)

print("CSV exported successfully")

CSV exported successfully
